## STEP 2 - TEXTUAL PREPROCESSING
#### In this step, we're coing to clean the texts before models training.

In [ ]:
from sklearn.model_selection import train_test_split                                                                                             
from pipeline.encoders.tfidf import TFIDFTokenizer                                                                                               
from pipeline.encoders.w2v import Word2VecTokenizer                                                                                              
import numpy as np                                                                                                                               
import pickle  
import pandas as pd                                                                                                                                  
import os     

In [20]:
df = pd.read_csv("data/augmented_datasets/mbti_augmented_qwen.csv")

In [21]:
df.head()

,Unnamed: 0,type,posts,augmented_posts
0,0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,1,ENTP,'I'm finding the lack of me in these posts ver...,I am noticing a significant absence of myself ...
2,2,INTP,'Good one _____ https://www.youtube.com/wat...,'Good one _____ https://www.youtube.com/wat...
3,3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","'Dear INTP, I enjoyed our conversation the o..."
4,4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc..."


In [22]:
pre = Preprocessor("augmented_posts")

In [23]:
df = pre.preprocess_complete(df)

In [24]:
df.head()

,Unnamed: 0,type,posts,augmented_posts,augmented_posts_clean
0,0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,moment sportscenter top ten play prank ha life...
1,1,ENTP,'I'm finding the lack of me in these posts ver...,I am noticing a significant absence of myself ...,i noticing significant absence post sex monoto...
2,2,INTP,'Good one _____ https://www.youtube.com/wat...,'Good one _____ https://www.youtube.com/wat...,good one course i say i know blessing curse do...
3,3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","'Dear INTP, I enjoyed our conversation the o...",dear i enjoyed conversation day esoteric gabbi...
4,4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc...",you fired another common misconception approac...


In [25]:
pre2 = Preprocessor("posts")
df = pre2.preprocess_complete(df)

In [26]:
df.head()

,Unnamed: 0,type,posts,augmented_posts,augmented_posts_clean,posts_clean
0,0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,moment sportscenter top ten play prank ha life...,moment sportscenter top ten play prank ha life...
1,1,ENTP,'I'm finding the lack of me in these posts ver...,I am noticing a significant absence of myself ...,i noticing significant absence post sex monoto...,i finding lack me post alarming sex boring pos...
2,2,INTP,'Good one _____ https://www.youtube.com/wat...,'Good one _____ https://www.youtube.com/wat...,good one course i say i know blessing curse do...,good one course i say i know blessing curse do...
3,3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","'Dear INTP, I enjoyed our conversation the o...",dear i enjoyed conversation day esoteric gabbi...,dear i enjoyed conversation day esoteric gabbi...
4,4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc...",you fired another common misconception approac...,you fired another silly misconception approach...


In [27]:
df.to_csv("data/clean_datasets/qwen_cleaned_dataset.csv", index = False)

In [28]:
df["posts_clean"] = df["posts_clean"].fillna("")                                                     
df["augmented_posts_clean"] = df["augmented_posts_clean"].fillna("")   

In [29]:
X_indices = df.index                                                                                                             
y = df["type"]                                                                                                                   
                                                                                                                           
indices_train, indices_test, y_train_orig, y_test = train_test_split(                                                            
    X_indices, y, test_size=0.2, random_state=42, stratify=y                                                                     
)      

In [30]:
X_test_text = df.loc[indices_test, "posts_clean"]                                                                                                
                                                                                                                                                
X_train_text_human = df.loc[indices_train, "posts_clean"]                                                                                        
y_train_human = y.loc[indices_train]                                                                                                             
                                                                                                                                               
mask_minority = y_train_human.str.contains('E|S')                                                                                                
indices_minority = indices_train[mask_minority]                                                                                                  
X_train_text_ai = df.loc[indices_minority, "augmented_posts_clean"]                                                                              
y_train_ai = y.loc[indices_minority]                                                                                                             
                                                                                                                                                  
X_train_text = pd.concat([X_train_text_human, X_train_text_ai])                                                                                  
y_train = pd.concat([y_train_human, y_train_ai])       

In [31]:
tfidf = TFIDFTokenizer()
X_train_tfidf = tfidf.fit_transform(X_train_text.values).toarray()
X_test_tfidf = tfidf.transform(X_test_text.values).toarray()

In [32]:
w2v = Word2VecTokenizer()
w2v.fit(X_train_text) 
X_train_w2v = X_train_text.apply(w2v.get_document_embedding).tolist()
X_test_w2v = X_test_text.apply(w2v.get_document_embedding).tolist()

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [33]:
with open(os.path.join("data/train_datasets", "X_train_tfidf.pkl"), "wb") as f: pickle.dump(np.stack(X_train_tfidf), f)
with open(os.path.join("data/test_datasets", "X_test_tfidf.pkl"), "wb") as f: pickle.dump(np.stack(X_test_tfidf), f)
with open(os.path.join("data/train_datasets", "X_train_w2v.pkl"), "wb") as f: pickle.dump(np.stack(X_train_w2v), f)
with open(os.path.join("data/test_datasets", "X_test_w2v.pkl"), "wb") as f: pickle.dump(np.stack(X_test_w2v), f)
with open(os.path.join("data/train_datasets", "y_train.pkl"), "wb") as f: pickle.dump(y_train.values, f)
with open(os.path.join("data/test_datasets", "y_test.pkl"), "wb") as f: pickle.dump(y_test.values, f)
with open(os.path.join("data/vectorizers", "tfidf_vectorizer.pkl"), "wb") as f: pickle.dump(tfidf, f)